# 章节实践 — 量化 Dispatch 算子开发

## 背景

在 [02.05 算子语义概览](02.05_operator_logic_overview.ipynb)、[02.06 核心流程拆解](02.06_dispatch_combine_core_flow.ipynb)、[02.07 Win 区布局](01.07_win_memory_layout.ipynb)、[02.08 Tiling 开发指引](02.08_tiling_guide.ipynb) 与 [02.09 Kernel Stage 开发指引](02.09_kernel_stage_guide.ipynb) 中，已经完整呈现了非量化版本 MoE Dispatch 算子的语义、Win 区布局、Tiling 字段与 Kernel 阶段划分。

在大模型推理场景下，token 数据从 BF16/FP16 量化到 FP8 可显著降低跨卡通信量。本章实践要求在已有非量化版本理解的基础上，**实现 MX 量化版本 Dispatch 算子的关键路径**。MX 量化（Modular Exponent，每 32 个值共用一个 8-bit 指数 scale，输出 FP8_E5M2）的算子核函数已在实践代码中提供完整 `Quant::ComputeMaxExp` / `ComputeScale` / `ComputeData` 实现，**不需要重复实现量化算法本身**，本章实践聚焦的是把量化嵌入到 Dispatch 的数据流中。

## 算子原型（量化版）

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">角色</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">名称</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">类型 / 形状</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">与非量化版差异</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输入</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>x</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>float16_t[bs, h]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">不变</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输入</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIds</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32_t[bs, k]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">不变</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandXOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>fp8_e5m2_t[bs*k, h]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>数据类型由 float16_t 改为 fp8_e5m2_t</strong></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>dynamicScalesOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>uint8_t[bs*k, ceil(h/32)]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>新增：每 32 个 FP8 值对应 1 个 e8m0 scale</strong></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>assistInfoOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32_t[bs*k, 3]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">不变</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertTokenNumsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int64_t[localExpertNum]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">不变</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>epSendCountsOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>int32_t[epWorldSize*localExpertNum]</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">不变</td>
  </tr>
</table>

Kernel 入口签名：

```cpp
__global__ __aicore__ __vector__ void DispatchKernel(
    __gm__ void* shmemSpace, GM_ADDR x, GM_ADDR expertIds,
    GM_ADDR expandXOut, GM_ADDR dynamicScalesOut, GM_ADDR assistInfoOut,
    GM_ADDR expertTokenNumsOut, GM_ADDR epSendCountsOut,
    GM_ADDR workspaceGM, DispatchTilingData tilingData);
```

## 实现要求

本实践需要补全 3 处函数体（位于 `include/moe_distribute_dispatch.h`），其余代码（Init / Process / CalCumSum / WaitAndFormatOutput 等）以及量化算法本身（`Mc2QuantKernel::MoeDistributeDispatchQuant`）已由脚手架提供。

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">TODO</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">函数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">涉及内容</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">TODO 1</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>MoeDistributeDispatch::SetTilingDataAndCal</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Buffer 大小计算：调用 <code>quantInst_.QuantInit(...)</code> 让量化模板算出 <code>hAlignSize_</code> / <code>hOutSizeAlign_</code> / <code>scaleOutBytes_</code> / <code>tokenQuantAlign_</code>；再推导 <code>blockCntPerToken_</code> / <code>hCommuSize_</code> / <code>aivUsedAllToAll_</code> 等派生量</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">TODO 2</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>MoeDistributeDispatch::TokenToExpertInQuant</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">发送流程：清零 → 搬入 → <code>quantInst_.QuantProcess(...)</code> 量化 → 填三元组 → 分两段 Copy 到 outTensor_ → <code>aclshmemx_mte_put_nbi</code> 非阻塞推送对端 Win数据区</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">TODO 3</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>MoeDistributeDispatch::CopyInAndOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">接收流程：从本卡 Win 整段搬入 → <code>quantInst_.CopyScalesToOut(...)</code> 拆出 scales 写到 <code>dynamicScalesOut</code> → 主体写到 <code>expandXOut</code> → 三元组写到 <code>assistInfoOut</code></td>
  </tr>
</table>

Tiling 简化、Host 主程序、Combine 核函数、测试数据生成与精度校验脚本全部沿用 CANN samples 仓 `Samples/2_Performance/moe_dispatch_and_combine_story` 的现有实现。

## 工程目录结构

本实践在 `practice_workspace/` 下构建独立工程：

```
practice_workspace/
├─ CMakeLists.txt                          # 构建脚本（沿用 sample）
├─ README.md                               # 沿用 sample
├─ include/
│  ├─ moe_distribute_dispatch.h           # ★ 本实践编辑文件：3 处 TODO 需补全
│  ├─ moe_distribute_dispatch_quant.h     # MX 量化模板（直接调用，无需修改）
│  ├─ moe_distribute_dispatch_non_quant.h # 非量化版参考
│  ├─ moe_distribute_combine.h            # Combine 核函数（沿用 sample）
│  └─ moe_distribute_comm.h               # 通信公共头
├─ src/
│  ├─ dispatch_and_combine_final.asc      # Host 主程序（沿用 sample）
│  ├─ 0_non_quant_naive.asc               # 非量化版主程序（参考）
│  └─ utils.h                             # 测试主程序 I/O 辅助
├─ scripts/
|   ├─ gen_data.py                         # 测试数据生成（沿用 sample）
|   └─ verify_result.py                    # 精度校验（沿用 sample）
└─ third_party/
    └─ shmem/                            # 占位目录，首次 cmake 时自动 git submodule update
```
在本示例目录运行本示例，运行过程中会在本示例目录生成：

- `input/`：输入 bin（按 `chip_{rankId}` 分目录）
- `golden/`：golden bin（用于精度对比）
- `output/`：算子输出 bin（按 `chip_{rankId}` 分目录）


本练习提供完整自包含脚手架 `scaffold_02_10/`（含 `CMakeLists.txt`、`include/`、`src/`、`cmake/`、`third_party/shmem/` 占位等）。运行下方第一个代码 cell 即把脚手架复制到 `practice_workspace/`，开发者只需编辑 `include/moe_distribute_dispatch.h` 中标有 `===== TODO N =====` 的 3 处函数体。

In [ ]:
# 工程初始化：把自包含脚手架 scaffold_01_10/ 复制到 practice_workspace/
import shutil
from pathlib import Path

SCAFFOLD = Path('./scaffold_02_10')
WORKSPACE = Path('practice_workspace')

assert SCAFFOLD.is_dir(), f'未找到脚手架目录：{SCAFFOLD.resolve()}'

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
shutil.copytree(SCAFFOLD, WORKSPACE)

print('脚手架已复制到', WORKSPACE.resolve())
print('请编辑', WORKSPACE / 'include' / 'moe_distribute_dispatch.h')
print('在文件中搜索 ===== TODO 1 / 2 / 3 ===== 定位 3 处待实现函数体。')

## Tiling 简化实现

本实践沿用 sample 仓的 `DispatchTilingData` 与 `SetDispatchTilingData`，开发者不需要修改。仅做一次回顾，重点理解 `symMemSize` 字段（量化版的 Win 区总大小，替代非量化版 `totalWinSizeEp`）：

```cpp
struct alignas(8) DispatchTilingData {
    uint32_t epWorldSize;
    uint32_t epRankId;
    uint32_t moeExpertNum;
    uint32_t bs;
    uint32_t k;
    uint32_t h;
    uint32_t expertTokenNumsType;   // 0: cumsum 模式 / 1: count 模式
    uint32_t aivNum;
    uint64_t symMemSize;            // 量化版：Win 区总大小（字节）
};

void SetDispatchTilingData(DispatchTilingData& d, int epWorldSize, int epRankId, int bs) {
    d.epWorldSize = epWorldSize;
    d.epRankId = epRankId;
    d.moeExpertNum = epWorldSize * 4;   // demo: 每卡 4 本地专家
    d.bs = bs;
    d.k = 8;                            // demo: topk = 8
    d.h = 7168;                         // demo: hidden = 7168
    d.expertTokenNumsType = 1;          // demo: count 模式
    d.aivNum = AIV_CORE_NUM;
    d.symMemSize = SHMEM_SPACE_SIZE;
}
```

Tiling 在 Kernel 的 Init 阶段通过 `SetTilingData(tilingData)`（已实现）拷贝到对象字段，再由 **TODO 1** 中的 `SetTilingDataAndCal` 推导派生量。

## TODO 1：Buffer 大小计算

> 🔗 [跳转到代码 TODO 1](./scaffold_02_10/include/moe_distribute_dispatch.h#L261)

**目标函数：** `MoeDistributeDispatch::SetTilingDataAndCal`（位于 `practice_workspace/include/moe_distribute_dispatch.h`）

**关键变量预声明（脚手架已声明，无需重新定义，直接赋值即可）：**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">变量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moeExpertNumPerRank_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每卡承载的本地专家数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIdsCnt_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>bs * k</code>，专家路由表条目数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hOutSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">单 token 的 FP8 主体字节数 = <code>axisH_ * sizeof(fp8_e5m2_t)</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hAlignSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">UB 内 token 搬入对齐到 128 数据后的字节数（由 QuantInit 写入）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hOutSizeAlign_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">UB 内 [FP8 主体 + scales + 三元组] 总对齐字节数（由 QuantInit 写入 + 末尾 +UB_ALIGN）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>scaleOutBytes_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">单 token 的 scales 字节数 = <code>Align2(Ceil32(axisH_)) * sizeof(fp8_e8m0_t)</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>tokenQuantAlign_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">三元组在 UB 内的 int32 偏移（由 QuantInit 写入）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>blockCntPerToken_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">单 token 的 480B 子块数量</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hCommuSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">单 token 在 Win数据区占用的字节数 = <code>blockCntPerToken_ * SPLIT_BLOCK_SIZE</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>axisHCommu_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>hCommuSize_</code> 换算成 FP8 元素个数</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertPerSizeOnWin_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每个对端 (epRankId, localExpertIdx) 槽位的总字节数 = <code>axisBS_ * hCommuSize_</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivUsedCumSum_</code> / <code>aivUsedAllToAll_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">后段核 / 前段核数量（与非量化版规则相同）</td>
  </tr>
</table>

**实现要点：** `QuantInit` 内部完成 Align256 / Align128 / Ceil32 等 MX 对齐计算，开发者只需调用即可拿到 4 个返回值；剩余派生量与非量化版一致。

**完成后**，建议在工程内补一个小型单测（或直接进入编译阶段），编译失败往往源自变量未初始化。

## UB 与 Win 区中 token 的拼接布局（量化前后对比）

理解 3 处 TODO 之前，先建立对单条 token 在内存中布局的直观印象。下图横向表示字节偏移，从左到右递增。

**非量化版（02.06–02.09 已介绍）**：UB 内 token 缓冲仅含 FP16 主体 + 三元组，按 480B 子块写入对端 Win 槽位的前 480B，每个 512B Win 槽位末尾 32B 是 token-flag 占位。

```
UB 内（一条 token）：
┌──────────────── FP16 主体 (axisH_ * 2B) ────────────────┬─ pad ─┬─ 三元组(12B+pad) ─┐
│  hOutSize_ = axisH_ * sizeof(float16_t)                  │   …   │ (epRankId,idx,k)  │
└──────────────────────────────────────────────────────────┴───────┴───────────────────┘

对端 Win数据区（单 token 占 blockCntPerToken_ 个 512B 槽位）：
┌─ 480B 数据子块 ─┬ 32B token-flag ┐┌─ 480B 数据子块 ─┬ 32B token-flag ┐ ...
```

**量化版（本章实践）**：UB 内 token 缓冲 = FP8 主体 + scales + 三元组三段相邻拼接；接收侧从同一槽位拆出 3 个目的张量。

```
UB 内（一条 token，对齐由 QuantInit 决定）：
┌──── FP8 主体 (axisH_ * 1B, Align256) ────┬── scales (scaleOutBytes_) ──┬── 三元组(12B+pad) ──┐
│  hOutSize_ = axisH_ * sizeof(fp8_e5m2_t)  │  每 32 个 FP8 → 1 个 e8m0   │ (epRankId,idx,k)    │
│  起始偏移 0                                │  起始偏移 Align256(axisH_)  │ 起始偏移            │
│                                            │                             │ tokenQuantAlign_*4  │
└────────────────────────────────────────────┴─────────────────────────────┴─────────────────────┘
          ↓                                              ↓                          ↓
     DataCopyPad → expandXOut[bs*k, h]   CopyScalesToOut → dynamicScalesOut   DataCopyPad → assistInfoOut

对端 Win数据区（与非量化版一致，按 480B + 32B token-flag 切槽位）：
┌─ 480B 子块(FP8+scales+三元组的切片) ─┬ 32B token-flag ┐ ... × blockCntPerToken_
```

**关键对应关系**：

- `hOutSize_`、`scaleOutBytes_`、`tokenQuantAlign_` 是接收侧 3 个 `DataCopyExtParams` 的 `blockLen` 来源（TODO 3 直接复用）。
- `Align256(axisH_)` 是 scales 在 UB 中的起始偏移，`CopyScalesToOut` 内部已使用，开发者无需手算。
- 发送侧 `Copy` 把 UB 的 480B 数据搬到 Win 槽位的前 480B，留出末尾 32B 给硬件写 token-flag。
- 单 token 在 Win 上占用 `blockCntPerToken_ * 512 B = hCommuSize_`，由 TODO 1 算出。

## TODO 2：发送流程

> 🔗 [跳转到代码 TODO 2](./scaffold_02_10/include/moe_distribute_dispatch.h#L357)

**目标函数：** `MoeDistributeDispatch::TokenToExpertInQuant`

**参数与脚手架声明：**

```cpp
__aicore__ inline void MoeDistributeDispatch::TokenToExpertInQuant(
    int32_t toRankId,                              // 目标卡 epRankId
    GlobalTensor<ExpandXOutType> dstWinGMTensor,   // 对端 Win数据区的当前槽位
    TQue<QuePosition::VECIN, 1> inQueue,           // VECIN 队列（UB）
    uint32_t srcTokenIndex,                        // 本卡待发送 token 的下标
    uint32_t fillExpertIdx,                        // 三元组中的目的 expertIdx
    uint32_t quantExpertIdx)                       // （保留，方便扩展）
```

**可直接调用的辅助函数（已封装）：**

<div align="left">

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">函数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">作用</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>quantInst_.QuantProcess(LocalTensor&lt;fp8_e5m2_t&gt;&amp; out, LocalTensor&lt;float16_t&gt;&amp; in)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">执行 MX 量化（含 ComputeMaxExp / ComputeScale / ComputeData）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>FillTriple(LocalTensor&lt;ExpandXOutType&gt;&amp; xOut, uint32_t tokenIndex, uint32_t k)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">把 (epRankId_, tokenIndex, k) 三元组追加到 token 末尾</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aclshmemx_mte_put_nbi(dst, src, count, peId, channel)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">非阻塞地把 UB 内容推送到对端 Win 区</td>
  </tr>
</table>

</div>

**关键易错点：**

- 量化前**必须**对 UB 内 token 缓冲清零（`Duplicate(uint8 视图, QUANT_PADDING_VALUE, Align128(axisH_) * sizeof(XType))`），否则 axisH_ 不是 128 的倍数时尾部脏数据会进入 ComputeMaxExp 影响 scale。
- `DataCopyPadExtParams<XType>` 第一项 `isPad` 在量化版必须设为 `true`。
- 两段 `Copy` 的步长 `{1, 1, 16, 15}` 表示每次迭代 dst 走 16 个 32B（=512B 槽位）、src 走 15 个 32B（=480B 数据段），用于把 480B 的连续数据填入 512B 的 Win 槽位的前 480B（后 32B 是 token-flag 占位）。
- `flagPadOffset_` 在两次调用之间需要按 `hCommuSize_ - flagPadOffset_` 翻转，对应 Win 区 ping-pong 半区。

## TODO 3：接收流程

> 🔗 [跳转到代码 TODO 3](./scaffold_02_10/include/moe_distribute_dispatch.h#L934)

**目标函数：** `MoeDistributeDispatch::CopyInAndOut`

**参数与脚手架声明：**

```cpp
__aicore__ inline void MoeDistributeDispatch::CopyInAndOut(
    LocalTensor<int32_t> xOutInt32Tensor,  // 已搬入 UB 的全部 token（int32 视图）
    GM_ADDR wAddr,                          // 本卡 Win数据区中当前 (源卡, 本地专家) 槽位起始地址
    uint32_t index,                         // 在 expertMapTensor_ / expertFinishNumTensor_ 中的下标
    uint32_t dstPosition,                   // 在 expandXOut 中的目标偏移（按 token 计）
    uint32_t arriveCount)                   // 本次确认到达并要整理的 token 条数
```

**可直接调用的辅助函数（已封装）：**

<div style="text-align: left;">

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">函数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">作用</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>quantInst_.CopyScalesToOut(dstPosition, scaleOutBytes_, xTmpTensor_, scalesCopyParams)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">从 <code>xTmpTensor_</code> 的 <code>Align256&lt;uint32_t&gt;(axisH_)</code> 偏移处取 scales，按 <code>scalesCopyParams</code> 拷出到 <code>dynamicScalesOut</code></td>
  </tr>
</table>

</div>

**搬入与拆解要点：**

- 一次 `DataCopyPad` 把 `arriveCount` 条 token 的全部 480B 数据块统一搬入 `xTmpTensor_`；`srcStride = UB_ALIGN = 32B` 用于跳过每个 512B 槽位末尾的 token-flag。
- `xTmpTensor_` 中每条 token 的布局：`[FP8 主体 hOutSize_ 字节][scales scaleOutBytes_ 字节][...][三元组 tokenQuantAlign_*sizeof(int32_t) 起]`。3 个 `DataCopyExtParams` 的 `blockLen` 分别对应 3 段；`srcStride` 用于跳过同一 token 内的其他段，到达下一 token 的同一段起点。
- `expandIdxGMTensor_` 即非量化版的 `assistInfoOut`，量化版语义不变。

完成 3 处 TODO 后，进入下一节进行编译验证。

## Kernel 入口与 Host 主程序

Kernel 入口 `DispatchKernel`、`CombineKernel` 以及 Host 主程序 `dispatch_and_combine_final.asc` 均沿用 sample 仓实现。Host 主程序的核心流程（已就绪、无需修改）：

```cpp
// shmem 初始化
ACL_CHECK_WITH_RET(aclshmemx_init_attr(ACLSHMEMX_INIT_WITH_DEFAULT, &attributes),
    ERROR_LOG("aclshmemx_init_attr failed"), return -1);
void *shmemSpace = aclshmem_align(DISPATCH_PACKAGE_SIZE, aclshmemSize);

// workspace 申请
uint8_t *disaptchWorkspaceGM;
ACL_CHECK(aclrtMalloc(reinterpret_cast<void**>(&disaptchWorkspaceGM),
                      16 * 1024 * 1024, ACL_MEM_MALLOC_HUGE_FIRST));

// Dispatch launch
DispatchKernel<<<AIV_CORE_NUM, nullptr, stream>>>(
    shmemSpace, xInDevice, expertIdsDevice,
    expandXOutDevice, dynamicScalesDevice, tokenSrcInfoDevice,
    expertTokenNumsDevice, sendCountsDevice,
    disaptchWorkspaceGM, dispatchTilingData);

// Combine launch（沿用 sample，无需关心）
CombineKernel<<<AIV_CORE_NUM, nullptr, stream>>>(
    shmemSpace, expandXInDevice, expertIdsDevice,
    tokenSrcInfoDevice, sendCountsDevice, expertScalesDevice,
    xOutDevice, combineTilingData);

ACL_CHECK(aclrtSynchronizeStream(stream));
```

## 编译

`practice_workspace/` 是完全自包含的 cmake 工程。`NPU_ARCH` 根据所用硬件设置，下示以 `dav-3510` 为例。首次编译会自动拉取 `third_party/shmem` 子仓。

In [ ]:
# 编译：practice_workspace 是完全自包含的 cmake 工程
import os
PRACTICE_DIR = os.path.abspath('practice_workspace')
BUILD_DIR = os.path.join(PRACTICE_DIR, 'build')

!source /usr/local/Ascend/ascend-toolkit/set_env.sh && \
    cmake -S {PRACTICE_DIR} -B {BUILD_DIR} -DNPU_ARCH=dav-3510 && \
    cmake --build {BUILD_DIR} --target moe_dispatch_and_combine_story -j

## 生成测试数据

沿用 sample 仓 `scripts/gen_data.py`，参数 `--quant-mode 4` 表示 MXFP8 动态量化模式。

In [ ]:
!cd practice_workspace && \
    python3 ./scripts/gen_data.py --chip-num-per-server 2 --bs 8 --quant-mode 4

## 运行（2 卡，bs = 8）

In [ ]:
import os
CANN_SAMPLES_ROOT = os.path.abspath(os.path.join('..', '..', '..', '..', 'cann-samples'))
BUILD_DIR = os.path.join(CANN_SAMPLES_ROOT, 'build_practice')
EXE = os.path.join(BUILD_DIR, 'Samples', '2_Performance',
                   'moe_dispatch_and_combine_story', 'moe_dispatch_and_combine_story')
!source /usr/local/Ascend/ascend-toolkit/set_env.sh && cd practice_workspace && {EXE} 2 8

## 精度校验

沿用 sample 仓 `scripts/verify_result.py`，逐条比对 `golden/` 与 `output/` 中 bin 文件的字节精确性。

In [ ]:
!cd practice_workspace && python3 ./scripts/verify_result.py

## 查看答案

完成实现并通过精度校验后，可对照参考答案学习每一处的关键写法。

In [ ]:
!cat answer/answer_02_10/01_buffer_size_calc.txt

In [ ]:
!cat answer/answer_02_10/02_send_flow.txt

In [ ]:
!cat answer/answer_02_10/03_receive_flow.txt

## 实践总结

本章实践基于 02.04–02.08 已建立的 Dispatch 算子认知，把 MX 量化嵌入到完整数据流中。完成 3 处 TODO 后，你应当对以下内容形成清晰记忆：

**1. Buffer 大小的来源**：MX 量化引入 FP8 主体（Align256）+ scales（Align2(Ceil32(h)) 个 e8m0）+ 三元组（UB_ALIGN）三段拼接，对齐计算由 `quantInst_.QuantInit` 统一负责；上层只需要按 480B 子块切到 Win 槽位即可。

**2. 发送侧的量化嵌入点**：MX 量化在 UB 内对每条 token 执行一次，结果与 scales 紧邻存放后整段 `aclshmemx_mte_put_nbi` 推送到对端 Win数据区。清零、`PipeBarrier`、两段 `Copy` 的 stride 是与非量化版的主要差异。

**3. 接收侧的拆解**：来自源卡的同一 token 槽位内拆出 FP8 主体、scales、三元组三段，分别写到 `expandXOut` / `dynamicScalesOut` / `assistInfoOut`。`CopyScalesToOut` 已经封装好 scales 在 UB 中的偏移逻辑。

**4. 与非量化版的对应关系**：核分工、Win 区 ping-pong、token-flag 自旋等待、`PipeBarrier<PIPE_ALL>` 汇合等模式完全不变；量化版只是替换了
。

**扩展练习建议**：

- 修改 `gen_data.py` 把 `--bs` 调到 32、`--h` 调到 4096，对比量化精度与带宽节省。
- 改造 `TokenToExpertInQuant`，把两段 `Copy` 合成一段（thinkpoint：stride 与对齐约束）。
- 阅读 `moe_distribute_dispatch_quant.h` 中 `ComputeMaxExp` / `ComputeScale` / `ComputeData` 的实现，理解 MX 量化为何选择共享指数模式。